In [ ]:
import torch
import numpy as np
import torchvision.transforms as transforms
from skimage.measure import regionprops, moments, moments_hu
import matplotlib.pyplot as plt

# Preprocessing functions
def mask_outside_bbox(mask, bbox):
    masked = torch.zeros_like(mask)
    x1, y1, x2, y2 = map(int, bbox)
    masked[y1:y2, x1:x2] = mask[y1:y2, x1:x2]
    return masked

def crop_mask(mask, bbox):
    x1, y1, x2, y2 = map(int, bbox)
    return mask[y1:y2, x1:x2]

def preprocess_masks(masks, bboxes, output_size=(64, 64)):
    processed_masks = []
    for i in range(masks.size(0)):
        # Visualization
        plt.imshow(masks[i], cmap='gray')
        plt.title(f'Mask original {i}')
        plt.show()
        
        masked = mask_outside_bbox(masks[i], bboxes[i])
        
        # Visualization
        plt.imshow(masked, cmap='gray')
        plt.title(f'Masked {i}')
        plt.show()
        
        cropped = crop_mask(masked, bboxes[i])
        
        # Visualization
        plt.imshow(cropped, cmap='gray')
        plt.title(f'Cropped {i}')
        plt.show()

        # Convert to uint8 type and multiply by 255
        cropped_uint8 = (cropped * 255).byte()
        
        transform = transforms.Compose([transforms.ToPILImage(),
                                        transforms.Resize(output_size),
                                        transforms.ToTensor()])
        resized = transform(cropped_uint8)
        processed_masks.append(resized)

        # Visualization
        plt.imshow(resized[0], cmap='gray')  # Resized tensor has shape (1, 64, 64), so we use [0] to visualize the image
        plt.title(f'Resized {i}')
        plt.show()

    return torch.stack(processed_masks, dim=0)


# Feature extraction functions
def extract_features(mask):
    mask_np = mask.cpu().squeeze().numpy()
    
    props = regionprops(mask_np.astype(int))[0]
    region_features = [
        props.area,
        props.centroid[0],
        props.centroid[1],
        props.eccentricity,
        props.solidity,
        props.extent,
        props.perimeter,
        props.major_axis_length,
        props.minor_axis_length
    ]

    m = moments(mask_np)
    hu = moments_hu(m)

    combined_features = region_features + hu.tolist()
    return np.array(combined_features)

def extract_features_from_masks(masks):
    feature_list = []
    for mask in masks:
        features = extract_features(mask)
        feature_list.append(features)
    return np.stack(feature_list)

# Sample usage
masks = torch.rand((3, 256, 256)) > 0.5  # example binary masks for 3 frames
bboxes = torch.tensor([[50, 50, 150, 150] for _ in range(3)])  # example bboxes for 3 frames

# Preprocess masks and extract features
processed_masks = preprocess_masks(masks, bboxes)
features = extract_features_from_masks(processed_masks)

print(features.shape)


In [ ]:
import torch

def get_shape_features_object_centric(feat):
    
    result = torch.zeros((12, 11, 16))

    num_obj = feat['num_obj']
    valid_masks = feat['sam_masks'][:num_obj]

    for i, v in enumerate(valid_masks):
        result[i, :, :] = extract_features(v)

    feat['object_centric_shape_feats'] = result
    return result